<a href="https://colab.research.google.com/github/lisssalima-tech/HW/blob/main/Hw3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import files
import os

#каталоги для промежуточных файлов
PROJECT_DIR = os.getcwd()
FRAMES_DIR = os.path.join(PROJECT_DIR, "frames")
OUT_FRAMES_DIR = os.path.join(PROJECT_DIR, "out_frames")

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUT_FRAMES_DIR, exist_ok=True)

uploaded = files.upload()

video_filename = next(iter(uploaded.keys()))
video_path = os.path.join(PROJECT_DIR, video_filename)

# На некоторых окружениях upload() не пишет файл сразу на диск
if not os.path.exists(video_path):
    with open(video_path, "wb") as f:
        f.write(uploaded[video_filename])

print("Загружено видео:", video_filename)
print("Путь к видео:", video_path)


Saving input.mp4 to input.mp4
Загружено видео: input.mp4
Путь к видео: /content/input.mp4


In [11]:
import cv2
#у меня часто вылезали ошибки в ланной части кода, поэтому я решила сделать, чтобы мне текстом выводились все проблемы
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Не удалось открыть видео: {video_path}")

fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
if fps <= 0:
    fps = 30.0
    print("FPS не считался из файла, используется запасной 30.0")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("FPS:", fps, "Frames:", frame_count)

i = 0
frame_step = 3
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if i % frame_step == 0:
        frame_file = os.path.join(FRAMES_DIR, f"{i:05d}.png")
        cv2.imwrite(frame_file, frame)
    i += 1

cap.release()
if i == 0:
    raise RuntimeError("Кадры не извлечены: проверьте формат видео и путь к файлу.")

print("Готово, кадров сохранено:", i)


FPS: 29.97002997002997 Frames: 185
Готово, кадров сохранено: 185


In [6]:
%pip install -U diffusers peft huggingface_hub
import torch
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
from diffusers import UniPCMultistepScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch_dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch_dtype
).to(device)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    print("xformers не включён (возможно, не установлен). Продолжаем без него.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recomme

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


xformers не включён (возможно, не установлен). Продолжаем без него.


In [7]:
import cv2
import numpy as np
from PIL import Image

def make_canny_image(image_pil):
    img = np.array(image_pil)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.Canny(img, 100, 200)
    img = img[:, :, None]
    img = np.concatenate([img, img, img], axis=2)
    return Image.fromarray(img)


In [ ]:
from tqdm.auto import tqdm
from PIL import Image

prompt = (
    "scene in the style of Arcane, painterly, dramatic lighting, "
    "hand-painted, detailed backgrounds, expressive characters"
)
negative_prompt = (
    "blurry, low quality, distorted, extra limbs, text, watermark, logo"
)

num_inference_steps = 20
guidance_scale = 7.5
controlnet_conditioning_scale = 1.0
strength = 0.6

frame_files = sorted(
    [f for f in os.listdir(FRAMES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
)
if not frame_files:
    raise RuntimeError("Не найдено кадров для обработки в frames/.")

for fname in tqdm(frame_files):
    in_path = os.path.join(FRAMES_DIR, fname)
    out_path = os.path.join(OUT_FRAMES_DIR, fname)
    if os.path.exists(out_path):
        continue

    init_image = Image.open(in_path).convert("RGB")
    control_image = make_canny_image(init_image)

    result = pipe(
        prompt=prompt,
        image=init_image,
        control_image=control_image,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=controlnet_conditioning_scale,
        negative_prompt=negative_prompt,
    )

    out_image = result.images[0]
    out_image.save(out_path)

print("Все кадры обработаны.")


  0%|          | 0/62 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


  0%|          | 0/12 [00:00<?, ?it/s]

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
import cv2
from PIL import Image
import os

out_video_path = os.path.join(PROJECT_DIR, "output_arcane.mp4")
files = sorted([f for f in os.listdir(OUT_FRAMES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
if not files:
    raise RuntimeError("Нет обработанных кадров в out_frames/.")

first_frame = Image.open(os.path.join(OUT_FRAMES_DIR, files[0]))
w, h = first_frame.size

fps_target = fps if "fps" in globals() and fps and fps > 1 else 15
fps_target = int(round(fps_target))
if fps_target < 1:
    fps_target = 15

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
videowriter = cv2.VideoWriter(out_video_path, fourcc, fps_target, (w, h))

for fname in files:
    frame = cv2.imread(os.path.join(OUT_FRAMES_DIR, fname))
    if frame is None:
        continue
    videowriter.write(frame)

videowriter.release()
print("Готово, видео сохранено в:", out_video_path)
